# 01 · Extracción de intervalos R-R desde PhysioNet
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valoo-1201/vfc-caos-som/blob/main/notebooks/01_extraccion_RR.ipynb)

**Responsable:** Integrante A · **Entrada:** PhysioNet · **Salida:** series NN en `DATA_DIR/rr/<base>/<registro>.npy` y `results/calidad_registros.csv`

Usamos las anotaciones de latido que ya traen las bases (no hace falta detectar picos R). Un intervalo NN cuenta solo si **ambos** latidos son normales (`N`).

In [ ]:
#@title 0 · Configuración (ejecutar siempre primero)
REPO = "Valoo-1201/vfc-caos-som"   # <- usuario/repositorio en GitHub
RAMA = "main"
USAR_DRIVE = True                  # guarda las series R-R en Google Drive para no descargarlas otra vez
MODO_DEMO = False                  # True = datos sintéticos (solo para probar el código sin internet)

import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    os.chdir("/content")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # Colab -> 🔑 Secretos (solo hace falta si el repo es privado)
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    if not os.path.exists("/content/vfc-caos-som"):
        subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/vfc-caos-som"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vfc-caos-som", "pull", "-q"], check=False)
    os.chdir("/content/vfc-caos-som")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/vfc-caos-som-data"
    else:
        DATA_DIR = "/content/data"
else:  # ejecución local: el notebook está en notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    DATA_DIR = "data"

sys.path[:0] = [os.path.abspath("src"), os.getcwd()]
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 3)
print("Repositorio:", os.getcwd(), "| Datos:", DATA_DIR, "| Demo:", MODO_DEMO)

In [ ]:
from hrvchaos import data, preprocess
from hrvchaos.plots import plot_poincare

BASES = ["nsr2db", "chf2db", "fantasia"]   # añadir "chfdb" y "nsrdb" para la validación externa
MAX_HORAS = 6        # leer solo las primeras horas de cada registro (None = todo). 6 h bastan para 6 ventanas de 2000 latidos
RR_DIR = os.path.join(DATA_DIR, "rr")

## Descarga (con caché)
La primera vez tarda unos minutos por base; después se lee desde Drive.

In [ ]:
registros = []
if MODO_DEMO:
    registros = data.synthetic_cohort(n_per_group=10)
else:
    for base in BASES:
        print(f"== {base} ==")
        registros += data.load_database(base, cache_dir=RR_DIR,
                                        max_seconds=MAX_HORAS * 3600 if MAX_HORAS else None)
resumen = data.records_summary(registros)
resumen.groupby(["database", "group"]).agg(sujetos=("record", "count"), horas_media=("hours", "mean"),
                                           rr_medio_s=("mean_rr_s", "mean"))

## Calidad de la señal
Porcentaje de intervalos descartados por el filtro fisiológico (0.3–2.0 s) y de mediana local (±20 %), y cuántas ventanas válidas de 2000 latidos aporta cada sujeto.

In [ ]:
filas = []
for r in registros:
    limpio, ok = preprocess.clean_rr(r.rr)
    ventanas = preprocess.segment(r.rr, ok, window=2000, max_windows=6)
    filas.append({"database": r.database, "record": r.record, "group": r.group, "n_nn": len(r.rr),
                  "pct_descartado": 100 * (1 - ok.mean()), "ventanas_validas": len(ventanas)})
calidad = pd.DataFrame(filas)
calidad.to_csv("results/calidad_registros.csv", index=False)
print("Sujetos sin ninguna ventana válida:", (calidad.ventanas_validas == 0).sum())
calidad.groupby("group")[["pct_descartado", "ventanas_validas"]].describe().round(2)

## Vistazo a las series
Tacograma (RR vs. latido) y diagrama de Poincaré de un sujeto de cada grupo.

In [ ]:
ejemplo = {}
for g in calidad.group.unique():
    r = next(x for x in registros if x.group == g)
    ejemplo[g] = preprocess.clean_rr(r.rr)[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for g, rr in ejemplo.items():
    axes[0].plot(rr[:1500] * 1000, lw=0.6, label=g)
axes[0].set_xlabel("Latido"); axes[0].set_ylabel("RR (ms)"); axes[0].set_title("Tacograma (primeros 1500 latidos)")
axes[0].legend()
plot_poincare(ejemplo, ax=axes[1])
plt.tight_layout(); plt.savefig("results/figures/01_tacograma_poincare.png", dpi=150); plt.show()

## Para discutir en equipo
- ¿Cuántos sujetos de cada grupo quedan con al menos 3 ventanas válidas?
- ¿El % descartado es mayor en ICC? (Esperable: más latidos ectópicos.) Anotarlo como limitación.
- La diferencia de edades entre nsr2db y chf2db: revisar en la documentación de PhysioNet y registrarla.